In [18]:
import os
import tarfile
import joblib
import boto3
import pandas as pd
import sagemaker
from sagemaker.estimator import Estimator

sess = sagemaker.Session()
region = boto3.Session().region_name
account = boto3.client("sts").get_caller_identity()["Account"]
role = sagemaker.get_execution_role()

print("region:", region)
print("account:", account)
print("role:", role)

region: us-east-2
account: 539609142303
role: arn:aws:iam::539609142303:role/SageMakerStudioExecutionRole2026


In [19]:
bucket_name = sess.default_bucket()
print("bucket_name:", bucket_name)

bucket_name: sagemaker-us-east-2-539609142303


In [20]:
os.chdir("/home/sagemaker-user/Prediccion_ventas_Equipo10")
print(os.getcwd())
print(os.path.exists("artifacts/data/monthly_clean.csv"))

/home/sagemaker-user/Prediccion_ventas_Equipo10
True


In [21]:
train_s3_uri = sess.upload_data(
    path="artifacts/data/monthly_clean.csv",
    bucket=bucket_name,
    key_prefix="data/train"
)

print(train_s3_uri)

s3://sagemaker-us-east-2-539609142303/data/train/monthly_clean.csv


In [22]:
image_name = "ml-predsales-byoc"
repository_uri = f"{account}.dkr.ecr.{region}.amazonaws.com/{image_name}"

print(repository_uri)

539609142303.dkr.ecr.us-east-2.amazonaws.com/ml-predsales-byoc


In [23]:
estimator = Estimator(
    image_uri=repository_uri + ":latest",
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket_name}/output",
    sagemaker_session=sess
)

In [25]:
import boto3
import json

ecr = boto3.client("ecr", region_name=region)

response = ecr.describe_repositories(repositoryNames=["ml-predsales-byoc"])
print(json.dumps(response["repositories"][0], indent=2, default=str))

{
  "repositoryArn": "arn:aws:ecr:us-east-2:539609142303:repository/ml-predsales-byoc",
  "registryId": "539609142303",
  "repositoryName": "ml-predsales-byoc",
  "repositoryUri": "539609142303.dkr.ecr.us-east-2.amazonaws.com/ml-predsales-byoc",
  "createdAt": "2026-04-12 04:02:12.736000+00:00",
  "imageTagMutability": "MUTABLE",
  "imageScanningConfiguration": {
    "scanOnPush": false
  },
  "encryptionConfiguration": {
    "encryptionType": "AES256"
  }
}


In [28]:
import boto3
import json

ecr = boto3.client("ecr", region_name=region)
response = ecr.describe_repositories(repositoryNames=["ml-predsales-byoc"])
print(json.dumps(response["repositories"][0], indent=2, default=str))

{
  "repositoryArn": "arn:aws:ecr:us-east-2:539609142303:repository/ml-predsales-byoc",
  "registryId": "539609142303",
  "repositoryName": "ml-predsales-byoc",
  "repositoryUri": "539609142303.dkr.ecr.us-east-2.amazonaws.com/ml-predsales-byoc",
  "createdAt": "2026-04-12 04:02:12.736000+00:00",
  "imageTagMutability": "MUTABLE",
  "imageScanningConfiguration": {
    "scanOnPush": false
  },
  "encryptionConfiguration": {
    "encryptionType": "AES256"
  }
}


In [33]:
from sagemaker.estimator import Estimator

estimator = Estimator(
    image_uri=repository_uri + ":latest",
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket_name}/output",
    sagemaker_session=sess
)

In [35]:
print("region actual:", region)
print("repository_uri actual:", repository_uri)
print("role actual:", role)

region actual: us-east-2
repository_uri actual: 539609142303.dkr.ecr.us-east-2.amazonaws.com/ml-predsales-byoc
role actual: arn:aws:iam::539609142303:role/SageMakerStudioExecutionRole2026


In [36]:
import boto3
import json

ecr = boto3.client("ecr", region_name=region)

try:
    response = ecr.describe_repositories(repositoryNames=["ml-predsales-byoc"])
    print("ECR ya existe")
    print(json.dumps(response["repositories"][0], indent=2, default=str))
except ecr.exceptions.RepositoryNotFoundException:
    response = ecr.create_repository(repositoryName="ml-predsales-byoc")
    print("ECR creado")
    print(json.dumps(response["repository"], indent=2, default=str))

ECR ya existe
{
  "repositoryArn": "arn:aws:ecr:us-east-2:539609142303:repository/ml-predsales-byoc",
  "registryId": "539609142303",
  "repositoryName": "ml-predsales-byoc",
  "repositoryUri": "539609142303.dkr.ecr.us-east-2.amazonaws.com/ml-predsales-byoc",
  "createdAt": "2026-04-12 04:02:12.736000+00:00",
  "imageTagMutability": "MUTABLE",
  "imageScanningConfiguration": {
    "scanOnPush": false
  },
  "encryptionConfiguration": {
    "encryptionType": "AES256"
  }
}


In [37]:
image_name = "ml-predsales-byoc"
repository_uri = f"{account}.dkr.ecr.{region}.amazonaws.com/{image_name}"

print(repository_uri)

539609142303.dkr.ecr.us-east-2.amazonaws.com/ml-predsales-byoc


In [39]:
from sagemaker.estimator import Estimator

estimator = Estimator(
    image_uri=repository_uri + ":latest",
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket_name}/output",
    sagemaker_session=sess
)

In [40]:
estimator.fit(
    {
        "train": f"s3://{bucket_name}/data/train"
    }
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: ml-predsales-byoc-2026-04-12-04-48-15-597


2026-04-12 04:48:17 Starting - Starting the training job...
2026-04-12 04:48:30 Starting - Preparing the instances for training...
2026-04-12 04:49:17 Downloading - Downloading the training image..2026-04-12 04:49:26,764 - train_sagemaker - INFO - Logger configurado correctamente. Log file: artifacts/logs/train_sagemaker_20260412_044926.log
2026-04-12 04:49:27,867 - train_sagemaker - INFO - Datos después de lags (dropna): 710,527 filas
2026-04-12 04:49:27,896 - train_sagemaker - INFO - Train: 693,022 | Val: 17,505 | last_block=33

2026-04-12 04:49:45 Training - Training image download completed. Training in progress.
2026-04-12 04:49:45 Uploading - Uploading generated training model
2026-04-12 04:49:45 Completed - Training job completed
Training seconds: 54
Billable seconds: 54


In [43]:
print("region actual:", region)

region actual: us-east-2


In [47]:
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large"
)

INFO:sagemaker:Creating model with name: ml-predsales-byoc-2026-04-12-05-16-06-698
INFO:sagemaker:Creating endpoint-config with name ml-predsales-byoc-2026-04-12-05-16-06-698
INFO:sagemaker:Creating endpoint with name ml-predsales-byoc-2026-04-12-05-16-06-698


----!

In [51]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

In [54]:
print(estimator.model_data)

s3://sagemaker-us-east-2-539609142303/output/ml-predsales-byoc-2026-04-12-04-48-15-597/output/model.tar.gz


In [55]:
import os
import tarfile
import joblib
import boto3
from urllib.parse import urlparse

model_s3_uri = estimator.model_data
print("model_s3_uri:", model_s3_uri)

parsed = urlparse(model_s3_uri)
model_bucket = parsed.netloc
model_key = parsed.path.lstrip("/")

local_tar = "/tmp/model.tar.gz"
extract_dir = "/tmp/model_artifact"

os.makedirs(extract_dir, exist_ok=True)

boto3.client("s3").download_file(model_bucket, model_key, local_tar)

with tarfile.open(local_tar) as tar:
    tar.extractall(path=extract_dir)

print("archivos extraídos:", os.listdir(extract_dir))

model = joblib.load("/tmp/model_artifact/model.joblib")
print("tipo de modelo:", type(model))

if hasattr(model, "feature_names_in_"):
    print("features esperadas:", list(model.feature_names_in_))
else:
    print("El modelo no tiene feature_names_in_")

model_s3_uri: s3://sagemaker-us-east-2-539609142303/output/ml-predsales-byoc-2026-04-12-04-48-15-597/output/model.tar.gz
archivos extraídos: ['model.joblib']


/tmp/ipykernel_584/2722150135.py:22: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)


tipo de modelo: <class 'sklearn.linear_model._ridge.Ridge'>
features esperadas: ['date_block_num', 'shop_id', 'item_id', 'item_category_id', 'lag_1', 'lag_2', 'lag_3', 'lag_mean_1_2']


In [56]:
feature_names = list(model.feature_names_in_)

sample = [{col: 0 for col in feature_names}]
sample

[{'date_block_num': 0,
  'shop_id': 0,
  'item_id': 0,
  'item_category_id': 0,
  'lag_1': 0,
  'lag_2': 0,
  'lag_3': 0,
  'lag_mean_1_2': 0}]

In [58]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

In [59]:
result = predictor.predict(sample)
print(result)

{'predictions': [-0.24812874781783867]}


In [60]:
predictor.delete_endpoint()
print("Endpoint eliminado")

INFO:sagemaker:Deleting endpoint configuration with name: ml-predsales-byoc-2026-04-12-05-16-06-698
INFO:sagemaker:Deleting endpoint with name: ml-predsales-byoc-2026-04-12-05-16-06-698


Endpoint eliminado
